# 03 · Post-training: SFT, LoRA and preferences
### BeliefLens · STAC AI Workshop · NYC · 21 October 2026

## Learning objectives
1. Compute a supervised fine-tuning objective
2. Explain a low-rank parameter update
3. Compute a DPO preference objective
4. Distinguish preference likelihood from calibration

All companies, numbers, labels and market observations in these notebooks are synthetic teaching examples. Offline outputs come from explicit deterministic rules, not an LLM. The same JavaScript case engine runs in the browser and through this Python client. Only a cell explicitly using `live=True` calls Together AI. No financial action is executed.

**Requirements:** Python 3.10+, Node.js 20+, Jupyter. No model weights or API key are needed for the default exercises. Run cells in order.

In [ ]:
from pathlib import Path
import sys, json, math, random, statistics
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_client.py").exists()), None)
if root is None:
    raise RuntimeError("Extract the complete workshop ZIP and open Jupyter inside its folder.")
if str(root) not in sys.path: sys.path.insert(0, str(root))
from lab_client import run, compact, save
print("Shared engine ready:", compact(run()))

## SFT: increase likelihood of demonstrated responses
For tokens $y_t$, SFT minimizes $-\sum_t\log p_\theta(y_t\mid x,y_{<t})$. Here we compute a miniature response loss. This is objective arithmetic, not language-model training.

In [ ]:
token_probabilities = [0.8, 0.6, 0.9, 0.7]
sft_loss = -sum(math.log(p) for p in token_probabilities) / len(token_probabilities)
print("Mean token cross-entropy:", round(sft_loss, 4))
assert sft_loss > 0

## LoRA: a constrained weight change
Instead of updating every entry of $W$, learn $\Delta W=(\alpha/r)BA$. This 2×3 example uses rank one. QLoRA additionally uses a quantized base model while training adapters; it changes memory requirements, not the need for evaluation.

In [ ]:
W = [[1., 0., -1.], [0.5, 1., 0.]]
A = [[0.1, -0.2, 0.3]]
B = [[0.4], [-0.1]]
delta = [[B[i][0]*A[0][j] for j in range(3)] for i in range(2)]
adapted = [[W[i][j]+delta[i][j] for j in range(3)] for i in range(2)]
print("Delta W:", delta)
print("Updated W:", adapted)
print("General parameter count: full m*n versus rank-r adapter r*(m+n)")

## DPO: a preference ratio relative to a reference
$L=-\log\sigma(\beta[\log\pi(y^+|x)-\log\pi(y^-|x)-\log\pi_{ref}(y^+|x)+\log\pi_{ref}(y^-|x)])$.

Optimize a two-response distribution. The “preferred” response is a label from a reviewer, not a truth certificate. RLHF commonly learns a reward from preferences and optimizes a policy; DPO directly optimizes a preference objective relative to a reference.

In [ ]:
sigmoid = lambda z: 1/(1+math.exp(-z))
beta, log_odds, learning_rate = 0.2, 0., 0.5
losses = []
for step in range(100):
    margin = beta * log_odds  # reference log-odds is zero
    losses.append(-math.log(sigmoid(margin)))
    gradient = -beta * (1-sigmoid(margin))
    log_odds -= learning_rate * gradient
print("DPO loss:", round(losses[0],3), "→", round(losses[-1],3))
print("Preferred response probability:", round(sigmoid(log_odds),3))
assert losses[-1] < losses[0]
print("This probability is NOT probability that the preferred response is true.")

## Learning check
A reviewer prefers a confident claim based on an outdated filing. Does reducing DPO loss fix the filing date? **No.** Preference data can reward confidence and preserve factual error. Grounding, calibration and permission checks require their own evidence.

**Optional real LLM experiment:** compare a base endpoint and an actual fine-tuned endpoint on identical held-out prompts through the local API. This package supplies no pretrained financial adapter; it never labels a prompt template as a trained model.

In [ ]:
# OPTIONAL: execute only after starting the local server with Together configured.
# Never save credentials in a notebook cell or output.
# from getpass import getpass
# workshop_key = getpass("Temporary workshop key: ")
# actual = run({"mode": "rag_tools"}, live=True, key=workshop_key)
# print(compact(actual))
# print(actual.get("usage"), actual.get("latency_ms"))

## What you have demonstrated
- **Compute a supervised fine-tuning objective:** Calculated token cross-entropy
- **Explain a low-rank parameter update:** Constructed BA and inspected its effect on W
- **Compute a DPO preference objective:** Optimized a genuine two-response preference loss
- **Distinguish preference likelihood from calibration:** Separated preferred language from empirically validated outcomes

**Exit ticket:** State one result, the evidence supporting it, and one limit that prevents you from claiming more. Save a run JSON or your numerical results as the experiment record.